In [ ]:
import os
import json
from tqdm import tqdm_notebook
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
import scipy
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split 
from sklearn.tree import DecisionTreeRegressor
import math
import warnings
warnings.filterwarnings("ignore")

مشروع فردي لإيكاترينا تشيبينيفا



## مقدمة
### 1 شرح الميزات والبيانات
https://www.kaggle.com/spscientist/students-performance-in-exams/home
تتكون مجموعة البيانات هذه من العلامات التي حصل عليها الطلاب في مختلف المواد.
الهدف من المشروع هو فهم تأثير العوامل المختلفة على أداء الطلاب والتنبؤ بمتوسط ​​الدرجات في ثلاث مواد



#### وصف البيانات
- الجنس: ذكر وأنثى
- العرق / العرق
- مستوى تعليم الوالدين
- الغداء: مستوى الغداء، قياسي أو مجاني/مخفض
- الدورة التحضيرية للاختبار: من المفترض أن "لا يوجد" تعني أن الطالب لم يحضر الدورات التحضيرية، و"مكتمل" تعني أنه أكملها.
- درجة الرياضيات
- درجة القراءة
- كتابة النتيجة
- **avg_score: الميزة المستهدفة، وهي متوسط الدرجات لثلاثة مواد**


In [ ]:
df = pd.read_csv('Downloads/StudentsPerformance.csv')
df.head()

In [ ]:
df['avg_score'] = (df['math score'] + df['reading score'] + df['writing score']) / 3
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
pd.unique(df['test preparation course'])

In [ ]:
pd.unique(df['parental level of education'])

In [ ]:
df.isnull().sum()


كما رأينا أعلاه، لا توجد قيم فارغة في إطار البيانات هذا


In [ ]:
plt.figure(figsize=(20, 10))
sns.heatmap(df.corr())

In [ ]:
plt.figure(figsize=(20, 10))
sns.boxplot(x='gender', y='avg_score', data=df)

In [ ]:
plt.figure(figsize=(20, 10))
sns.boxplot(y='avg_score', x='test preparation course', hue='gender', data=df)

In [ ]:
plt.figure(figsize=(20, 10))
sns.countplot(x='race/ethnicity', data=df)

In [ ]:
plt.figure(figsize=(20, 10))
sns.countplot(x='parental level of education', data=df)

In [ ]:
sns.pairplot(df)

In [ ]:
pd.unique(df['parental level of education'])

In [ ]:
sns.barplot(x='avg_score', hue='parental level of education', data=df)

In [ ]:
# Function that prints summary statistics of column given in parameters
def summary_statistics(col, df):
    print('Mean: {}'.format(df[col].mean()))
    print('Max: {}'.format(df[col].max()))
    print('Min: {}'.format(df[col].min()))
    print('Median: {}'.format(df[col].median()))
    print()
    
    # Number of students with maximum and minimum score
    max_score = df[col].max()
    min_score = df[col].min()
    print("Number of students who scored maximum score: {}".format(df[col][df[col]==max_score].count()))
    print("Number of students who scored minimum score: {}".format(df[col][df[col]==min_score].count()))
    print()
    
    # Students close to mean i.e. Students that have scores equal to floor(mean score) or ceiling(mean score)
    near_mean_floor = math.floor(df[col].mean())
    near_mean_ceil = math.ceil(df[col].mean())
    near_mean_tot = df[col][df[col]==near_mean_floor].count() + df[col][df[col]==near_mean_ceil].count()
    print("Number of students close to mean score: {}".format(near_mean_tot))
    print()
    
    # Students that have 50th percentile
    print("Number of students at median score: {}".format(df[col][df[col]==df[col].median()].count()))
    
    # Students with 25th percentile and 75th percentile scores
    print("Number of students at 25th percentile: {}".format(df[col][df[col]==df[col].quantile(0.25)].count()))
    print("Number of students at 75th percentile: {}".format(df[col][df[col]==df[col].quantile(0.75)].count()))

In [ ]:
summary_statistics("math score", df)

In [ ]:
summary_statistics("reading score", df)

In [ ]:
summary_statistics("writing score", df)


**هل يؤثر إكمال الدورة التدريبية فعليًا على النتيجة؟**


In [ ]:
#Students that have more than median marks in 

# Maths
print("Maths")
df_top_math = df[df["math score"] > df["math score"].median()]
print(df_top_math["test preparation course"].value_counts())
print()

# Reading
print("Reading")
df_top_read = df[df["reading score"] > df["reading score"].median()]
print(df_top_read["test preparation course"].value_counts())
print()

# Writing
print("Writing")
df_top_writ = df[df["writing score"] > df["writing score"].median()]
print(df_top_writ["test preparation course"].value_counts())
print()

print("Average score")
df_top_writ = df[df["avg_score"] > df["avg_score"].median()]
print(df_top_writ["test preparation course"].value_counts())
print()


يمكننا أن نرى أن هناك نسبة 1:1 تقريبًا للطلاب الذين أكملوا الدورة والطلاب الذين لم يكملوا الدورة (لأكثر من متوسط الدرجات). لذلك، يمكننا وضع فرضية مثل أن إمكانية الحصول على درجات أكثر من المتوسط ​​في الاختبار تظل دون تغيير بعد الانتهاء.
لكن هذا لا يمكن أن يقال كنتيجة نهائية.


In [ ]:
#Students that have less than or equal to median marks in  

# Maths
print("Maths")
df_bot_math = df[df["math score"] <= df["math score"].median()]
print(df_bot_math["test preparation course"].value_counts())
print()

# Reading
print("Reading")
df_bot_read = df[df["reading score"] <= df["reading score"].median()]
print(df_bot_read["test preparation course"].value_counts())
print()

# Writing
print("Writing")
df_bot_writ = df[df["writing score"] <= df["writing score"].median()]
print(df_bot_writ["test preparation course"].value_counts())
print()

print("Average score")
df_top_writ = df[df["avg_score"] <= df["avg_score"].median()]
print(df_top_writ["test preparation course"].value_counts())
print()


بالنسبة للطلاب الذين حصلوا على درجات أقل من أو تساوي المتوسط، فقد ثبت خطأ الفرضية المفترضة أعلاه. حيث أن هناك نسبة 2:1 للطلاب الذين لم يكملوا الدورة إلى الطلاب الذين أكملوا الدورة. لذا فإن فرضيتنا الجديدة يمكن أن تكون كذلك
من المرجح أن تحصل على درجات أقل من المتوسط إذا لم تكمل الدورة التحضيرية للاختبار
لكن هذا أيضاً لا يمكن اعتباره استنتاجاً نهائياً.


In [ ]:
def graphs(score_type, suptitle, groupbyterm, kind):
    nrows = 2
    ncols = 3
    inches = 5
    df_female = df[df['gender'] == 'female']
    df_male = df[df['gender'] == 'male']
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*inches,nrows*inches))
    fig.suptitle(suptitle)
    temp = df[groupbyterm].value_counts().rename("")
    temp.plot.pie(ax=axes[0,0], title="Students Overall", autopct="%.2f", legend=False)
    temp = df_female[groupbyterm].value_counts().rename("")
    temp.plot.pie(ax=axes[0,1], title="Female", autopct="%.2f", legend=False)
    temp = df_male[groupbyterm].value_counts().rename("")
    temp.plot.pie(ax=axes[0,2], title="Male", autopct="%.2f", legend=False)
    pd.concat([
        df_female.groupby(groupbyterm)[score_type].mean().rename("Female"),
        df_male.groupby(groupbyterm)[score_type].mean().rename("Male")], axis=1).plot(kind="bar", ax=axes[1,0], legend=True)
    axes[1,0].set_xlabel("")
    axes[1,0].legend([x.get_text().capitalize() for x in axes[1,0].legend().get_texts()])
    axes[1,0].set_xticklabels([x.get_text().capitalize() for x in axes[1,0].get_xticklabels()])
    df_female.groupby(groupbyterm)[score_type].plot(kind=kind, ax=axes[1,1], legend=True, alpha=0.8, histtype="step")
    axes[1,1].set_xlabel("Female Scores")
    axes[1,1].set_ylabel("")
    axes[1,1].set_xticks(np.arange(0, 101, step=20))
    axes[1,1].set_yticks(np.arange(0, 101, step=20))
    axes[1,1].legend([x.get_text().capitalize() for x in axes[1,1].legend().get_texts()])
    df_male.groupby(groupbyterm)[score_type].plot(kind=kind, ax=axes[1,2], legend=True, alpha=0.8, histtype="step")
    axes[1,2].set_xlabel("Male Scores")
    axes[1,2].set_ylabel("")
    axes[1,2].set_xticks(np.arange(0, 101, step=20))
    axes[1,2].set_yticks(np.arange(0, 101, step=20))
    axes[1,2].legend([x.get_text().capitalize() for x in axes[1,2].legend().get_texts()])
    return fig, axes

In [ ]:
fig, axes = graphs("avg_score","Avg Scores by Parental Level of Education", "parental level of education", "hist")

عدد قليل جداً من الآباء يحملون درجة البكالوريوس؛ وعدد أقل من الحاصلين على درجة الماجستير. ونظرًا لأن لدينا عددًا أقل من أولياء الأمور الحاصلين على درجات علمية في الرياضيات، فمن الواضح أن درجات هؤلاء الطلاب ستكون أعلى في المتوسط، مقارنةً بالدرجات الأخرى. لكن بالنظر إلى الدرجات الأكثر كثافة سكانية، تظهر نتائجها أيضًا اختلافات، وإن لم تكن كثيرًا، ولكنها تظهر اختلافات مع ذلك.


In [ ]:
fig, axes = graphs("avg_score","Avg Scores by Race/Ethnicity", "race/ethnicity", "hist")


أقصر عدد من السكان في العرق/المجموعة العرقية أ. ومن حيث الأداء، لا يزالون في أدنى مستوياتهم في المتوسط. السكان الأكثر كثافة في العرق/المجموعة العرقية ج. من حيث الأداء، فهم متوسطون.


In [ ]:
fig, axes = graphs("avg_score","Avg Scores by Lunch", "lunch", "hist")


وباتباع المتوسطات، لا يزال الأولاد يتفوقون على البنات في الرياضيات بغض النظر عن الغداء. تناول ما يقرب من ثلثي الطلاب وجبة غداء عادية. من الجيد أن نعرف أن هؤلاء الطلاب لم يضحوا ببطونهم من أجل النتائج.


In [ ]:
fig, axes = graphs("avg_score","Avg Scores by Test Preparation Course", "test preparation course", "hist")


انطلاقًا من الأرقام الموجودة في الرسوم البيانية، فإن ثلث الطلاب فقط يأخذون هذه الدورات على محمل الجد. وأولئك الذين أخذوا الدورات لا يتمتعون بميزة كبيرة في الدرجات. فهل يجب إلغاء الدورات التحضيرية؟ وبالنظر إلى الحدود الأصغر، فإن الطلاب الذين أخذوا الدورة التحضيرية لديهم خطوط أساس أعلى من أولئك الذين لم يفعلوا ذلك. لذلك ربما يكون من الجيد الاستمرار في ذلك.



** توزيع درجات الطلاب **
يتم توزيع جميع الدرجات بشكل طبيعي تقريبًا. تُظهر مخططات Q-Q التواءً في كلا الاتجاهين، مما يشير إلى الانحراف عن التوزيع الطبيعي في تلك المناطق.
تُظهر التوزيعات المشتركة ارتباطًا قويًا بين درجات الاختبار بين المواد المختلفة وهو أمر ليس مفاجئًا.


In [ ]:
score_cols = ['math score', 'reading score','writing score', 'avg_score']
from scipy.stats import norm


def Plot_Dist(df, col):
    fig,axarr = plt.subplots(1,2,figsize=(12,4))
    # plot distribution
    sns.distplot(df[col], fit=norm, kde=False, ax=axarr[0])
    #Q-Q plot
    from statsmodels.graphics.gofplots import qqplot
    qqplot(df['math score'],line='s', ax=axarr[1])
    fig.suptitle(col+' distribution', fontsize=14)
    plt.show()

Plot_Dist(df,col='math score')
Plot_Dist(df,col='reading score')
Plot_Dist(df,col='writing score')
Plot_Dist(df,col='avg_score')


ax1=sns.jointplot(x="math score", y="reading score", data=df)
plt.show()

ax2=sns.jointplot(x="math score", y="writing score", data=df)
plt.show()


**تباين الدرجات مع جنس الطالب**
تتفوق الطالبات على نظرائهن من الذكور في القراءة والكتابة. وفي الرياضيات، يكون أداء الأولاد في المتوسط ​​أفضل من أداء البنات.


In [ ]:
def Plot_Set(df, xcol, ycols):
    df = df.sort_values(by=xcol)
    fig,axarr = plt.subplots(1, 4,figsize=(12,5))
    for id,ycol in enumerate(ycols):
        medians = df.groupby([xcol])[ycol].median().values
        median_labels = [str(np.round(s, 2)) for s in medians]
        pos = range(len(medians))
        sns.boxplot(x=xcol, y=ycol, data=df, width=0.5, palette='Set3', ax=axarr[id], linewidth=0.5)
        for tick,label in zip(pos,axarr[id].get_xticklabels()):
            axarr[id].text(pos[tick], medians[tick] + 0.5, median_labels[tick], horizontalalignment='center', size='medium', color='k', weight='semibold')
        axarr[id].set_ylim([0,105])
        plt.setp(axarr[id].get_xticklabels(), rotation=25,ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
Plot_Set(df, xcol='gender', ycols=['math score','reading score','writing score', 'avg_score'])

** تباين النتائج مع العرق / العرق **
العرق له تأثير كبير على درجات الاختبار. بالنسبة لجميع المواد، يكون أداء الطلاب في المجموعة E أفضل من الطلاب من العرقيات الأخرى.


In [ ]:
Plot_Set(df,xcol='race/ethnicity',ycols=['math score','reading score','writing score', 'avg_score'])


**اختلاف الدرجات مع مستوى تعليم الوالدين**
المستوى التعليمي للوالدين له تأثير مباشر على درجات الاختبار. ارتفاع المستوى التعليمي للوالد، وارتفاع درجات الطلاب.


In [ ]:
Plot_Set(df,xcol='parental level of education',ycols=['math score','reading score','writing score', 'avg_score'])


** تباين النتائج مع نوع الغداء **
إن حصول الطالب على وجبة الغداء القياسية أو وجبة الغداء المجانية/المخفضة له تأثير على النتائج. ومن الواضح أن الطلاب من الأسر ذات الدخل المنخفض يحصلون في المتوسط ​​على درجات أقل بمقدار 5 إلى 10 نقاط من أولئك الذين يستطيعون تحمل تكاليف وجبات الغداء القياسية.


In [ ]:
Plot_Set(df,xcol='lunch',ycols=['math score','reading score','writing score', 'avg_score'])


# التنبؤ 
يعد MAE وRMSE المقياسين الأكثر شيوعًا للمتغيرات المستمرة. لنبدأ مع الأكثر شعبية.
من السهل تفسير MAE لأنه يأخذ مباشرة متوسط ​​الإزاحات بينما يعاقب RMSE الفرق الأعلى أكثر من MAE. ولذلك، اخترت mse كمقياس
تم اختيار شجرة القرار كنموذج للتنبؤ.تقنيات تصنيف الأشجار، عندما "تعمل" وتنتج تنبؤات دقيقة أو تصنيفات متوقعة بناءً على عدد قليل من الشروط المنطقية، تتمتع بعدد من المزايا مقارنة بالعديد من تلك التقنيات البديلة. بساطة النتائج. في معظم الحالات، يكون تفسير النتائج الملخصة في شجرة بسيطًا جدًا. هذه البساطة مفيدة ليس فقط لأغراض التصنيف السريع للملاحظات الجديدة (من الأسهل بكثير تقييم شرط منطقي واحد أو اثنين فقط، بدلاً من حساب درجات التصنيف لكل مجموعة محتملة، أو قيم متوقعة، بناءً على جميع المتنبئين وربما استخدام بعض معادلات النماذج غير الخطية المعقدة)، ولكن يمكن أيضًا أن تسفر في كثير من الأحيان عن "نموذج" أبسط بكثير لشرح سبب تصنيف الملاحظات أو التنبؤ بها بطريقة معينة (على سبيل المثال، عند تحليل مشاكل العمل، يكون من الأسهل بكثير تقديم بعض عبارات "إذا" البسيطة إلى الإدارة، من بعض المعادلات المعقدة).
للمعالجة المسبقة للبيانات، يتم استخدام ترميز ساخن واحد للميزات الفئوية.

In [ ]:
df.head()

In [ ]:
df = pd.concat([df, pd.get_dummies(df.gender, prefix='gender_')], axis=1)
df = pd.concat([df, pd.get_dummies(df['race/ethnicity'], prefix='race_')], axis=1)
df = pd.concat([df, pd.get_dummies(df['parental level of education'], prefix='edu_')], axis=1)
df = pd.concat([df, pd.get_dummies(df['lunch'], prefix='lunch_')], axis=1)
df = pd.concat([df, pd.get_dummies(df['test preparation course'], prefix='course_')], axis=1)

In [ ]:
df.drop(columns=['gender', 'race/ethnicity','parental level of education', 'lunch', 'test preparation course'], inplace=True)

In [ ]:
target = df.avg_score
df.drop(columns=['math score', 'reading score', 'writing score', 'avg_score'], inplace=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.33, random_state=17)

In [ ]:
dtr = DecisionTreeRegressor(max_depth=4, random_state=17)

cross_val_score(dtr, df, target, cv=10, scoring='neg_mean_squared_error')

In [ ]:
dtr = DecisionTreeRegressor(max_depth=4, random_state=17)

dtr.fit(X_train, y_train)

y_pred = dtr.predict(X_test)
mean_absolute_error(y_test, y_pred)